# 06 tokenizer、chat template 和输入边界

目标：把文本进入模型前最容易踩坑的部分拆开实践：普通文本、chat template、special tokens、padding、truncation、attention mask，以及 SFT 训练时的 label mask。

面试里这类问题经常不是问 API 名字，而是问你是否知道输入格式错了会让模型表现突然变差。


## 运行环境准备

默认模型仍然使用 `Qwen/Qwen2.5-0.5B-Instruct`。如果你在本地已经下载了模型，也可以把环境变量 `MODEL_ID` 指向本地目录。


In [ ]:
from pathlib import Path

requirements_path = Path("requirements.txt")
if not requirements_path.exists():
    requirements_path = Path("../requirements.txt")

%pip install -r {requirements_path}


In [ ]:
import os
from pathlib import Path

MODEL_ID = os.getenv("MODEL_ID", "Qwen/Qwen2.5-0.5B-Instruct")
MODEL_SOURCE = os.getenv("MODEL_SOURCE", "modelscope").lower()


def resolve_model_path(model_id):
    if Path(model_id).exists():
        return model_id
    if MODEL_SOURCE != "modelscope":
        return model_id

    from modelscope import snapshot_download
    return snapshot_download(model_id)


MODEL_PATH = resolve_model_path(MODEL_ID)
print("MODEL_ID =", MODEL_ID)
print("MODEL_PATH =", MODEL_PATH)


## 1. 先看 tokenizer 的关键属性

重点关注：词表大小、最大长度、special tokens、pad/eos 的关系，以及 chat template 是否存在。


In [ ]:
from transformers import AutoTokenizer


tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("tokenizer class:", type(tokenizer).__name__)
print("vocab_size:", tokenizer.vocab_size)
print("model_max_length:", tokenizer.model_max_length)
print("padding_side:", tokenizer.padding_side)
print("special_tokens_map:", tokenizer.special_tokens_map)
print("pad_token_id:", tokenizer.pad_token_id)
print("eos_token_id:", tokenizer.eos_token_id)

chat_template = getattr(tokenizer, "chat_template", None)
print("has chat_template:", bool(chat_template))
if chat_template:
    print(chat_template[:1200])


## 2. 普通文本和 chat template 的 token 数对比

Instruct 模型训练时通常看到的是带角色标记的对话格式。生产里如果直接把用户文本丢进去，模型可能还能答，但行为会不稳定。


In [ ]:
messages = [
    {"role": "system", "content": "你是一个严谨的模型部署面试官。"},
    {"role": "user", "content": "用三句话解释 tokenizer、embedding 和 logits 的关系。"},
]

plain_text = messages[-1]["content"]
chat_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

plain_ids = tokenizer(plain_text)["input_ids"]
chat_ids = tokenizer(chat_text)["input_ids"]

print("plain text:\n", plain_text)
print("\nchat template text:\n", chat_text)
print("\nplain token count:", len(plain_ids))
print("chat token count:", len(chat_ids))
print("\nfirst 30 chat ids:", chat_ids[:30])
print("first 30 chat tokens:", tokenizer.convert_ids_to_tokens(chat_ids[:30]))


## 3. special tokens 会影响 decode 结果

`skip_special_tokens=True` 适合展示给用户；调试 prompt 边界时要关掉它，否则看不到角色标记和结束符。


In [ ]:
print("decode with special tokens:")
print(tokenizer.decode(chat_ids[:80], skip_special_tokens=False))

print("\ndecode for user display:")
print(tokenizer.decode(chat_ids[:80], skip_special_tokens=True))


## 4. padding、truncation 和 attention mask

面试常问：为什么 batch 推理要 pad？为什么 decoder-only 模型常用 left padding？attention mask 不对会发生什么？


In [ ]:
examples = [
    "短问题：什么是 KV cache？",
    "长一点的问题：请解释 decoder-only 语言模型在 prefill 和 decode 两个阶段里分别做了什么。",
]

for padding_side in ["right", "left"]:
    tokenizer.padding_side = padding_side
    batch = tokenizer(
        examples,
        padding=True,
        truncation=True,
        max_length=32,
        return_tensors="pt",
    )
    print("=" * 80)
    print("padding_side =", padding_side)
    print("input_ids shape:", tuple(batch["input_ids"].shape))
    print("attention_mask shape:", tuple(batch["attention_mask"].shape))
    print("input_ids:")
    print(batch["input_ids"])
    print("attention_mask:")
    print(batch["attention_mask"])
    print("decoded row 0:")
    print(tokenizer.decode(batch["input_ids"][0], skip_special_tokens=False))

# 恢复成生成更常用的 left padding。
tokenizer.padding_side = "left"


## 5. SFT 训练时只监督 assistant 部分

监督微调时通常把 prompt token 的 label 置为 `-100`，只让 assistant answer 参与 loss。否则模型会被训练去复述 system/user prompt。


In [ ]:
train_messages = [
    {"role": "system", "content": "你是一个部署专家。"},
    {"role": "user", "content": "解释 TTFT 和 TPOT 的区别。"},
    {"role": "assistant", "content": "TTFT 是首 token 延迟，TPOT 是后续每个 token 的平均生成耗时。"},
]

prompt_messages = train_messages[:-1]
prompt_text = tokenizer.apply_chat_template(
    prompt_messages,
    tokenize=False,
    add_generation_prompt=True,
)
full_text = tokenizer.apply_chat_template(
    train_messages,
    tokenize=False,
    add_generation_prompt=False,
)

prompt_ids = tokenizer(prompt_text)["input_ids"]
full_ids = tokenizer(full_text)["input_ids"]
labels = [-100] * len(full_ids)
labels[len(prompt_ids):] = full_ids[len(prompt_ids):]

print("prompt token count:", len(prompt_ids))
print("full token count:", len(full_ids))
print("supervised token count:", sum(x != -100 for x in labels))
print("\nfirst supervised text:")
answer_ids = [token_id for token_id, label in zip(full_ids, labels) if label != -100]
print(tokenizer.decode(answer_ids, skip_special_tokens=False))


## 面试总结

- tokenizer 不是简单按字切分，它决定了输入 token 边界、长度和最终成本。
- instruct/chat 模型要使用训练时匹配的 chat template。
- `eos_token_id` 控制自然停止，`pad_token_id` 控制 batch padding，二者可以相同但语义不同。
- batch 推理必须配正确的 `attention_mask`，否则 padding token 可能被模型当成有效上下文。
- SFT 通常只对 assistant answer 计算 loss，prompt 部分 label 置为 `-100`。
- 排查线上回答异常时，第一步应该打印最终 prompt、token 数、special tokens 和截断位置。
